# Backward particle tracking

This notebook shows how to run **backward particle tracking** to find the **capture zone** of a pumping well: the area of the aquifer whose water eventually reaches the well. Instead of following water forward from a source, you release particles at the well and track them *upstream* to see where that water came from.

By the end you will be able to build a groundwater-flow (**GWF**) model, drive two different particle-tracking codes from its output, and post-process the tracks into a capture area and a travel-time distribution. The flow system is taken from [an example](https://modflow6-examples.readthedocs.io/en/develop/_notebooks/ex-prt-mp7-p01.html) originally published in the MODPATH 7 user guide.

This notebook builds **three** simulations in sequence:

1. a **flow** (**GWF**) model, run on its own;
2. a **MODPATH 7** (**MP7**) tracking model — the long-established stand-alone code — driven by the flow output; and
3. a MODFLOW 6 **particle-tracking (PRT)** model driven by the same output.

Running MP7 and PRT side by side shows how to migrate an MP7 model to PRT and how the two codes compare.

## Phase 1 — Build and run the flow model

Start with the units. MODFLOW has no built-in unit system, so every input must share one consistent length and time unit. Set them once here — **days** for time and **feet** for length — and use them everywhere.

In [ ]:
time_unit = "days"
length_unit = "feet"

Set up the time discretization. Each entry in `tdis_pd` is a tuple `(perlen, nstp, tsmult)` — the period length, number of time steps, and time-step multiplier. This simulation has a single steady-state stress period of length `perlen`.

In [ ]:
nstp = 1
perlen = 1000.0
tsmult = 1.0
tdis_pd = [(perlen, nstp, tsmult)]
nper = len(tdis_pd)

Set up the grid discretization. This is a three-layer model (`nlay`, `nrow`, `ncol`) with two relatively permeable layers sandwiching a narrow confining bed. `delr` and `delc` are the column and row widths; `top` and `bot` give the top elevation and the bottom of each layer.

In [ ]:
nlay, nrow, ncol = 3, 21, 20
Lx, Ly = 10000.0, 10500.0
delr, delc = Lx / ncol, Ly / nrow
top = 400.0
bot = [220.0, 200.0, 0.0]

Set up the workspaces — the folders where model files are written. Use `pathlib.Path` so paths work on any operating system. Create a base workspace for the whole scenario and a nested `gwf` workspace for the flow model, so each of the three simulations keeps its files separate.

In [ ]:
from pathlib import Path

example_name = "prt_backward"
gwf_name = f"{example_name}-gwf"
base_ws = Path("models") / example_name
gwf_ws = base_ws / "gwf"
gwf_ws.mkdir(exist_ok=True, parents=True)

#### Create the flow simulation and model

Build the flow simulation from the top down. Create the simulation container with `flopy.mf6.MFSimulation()`, add the temporal discretization (**TDIS**) with `flopy.mf6.ModflowTdis()`, then create the groundwater-flow model (`gwf`) with `flopy.mf6.ModflowGwf()`. Inside the model, add the spatial discretization (**DIS**) package with `flopy.mf6.ModflowGwfdis()` and the initial conditions (**IC**) package with `flopy.mf6.ModflowGwfic()`, setting the starting head (`strt`) to the model top. Note the nesting: the simulation is passed as the first argument to `ModflowTdis` and `ModflowGwf`, and the model is passed as the first argument to each of its packages.

In [ ]:
import flopy

gwf_sim = flopy.mf6.MFSimulation(
    sim_name=gwf_name,
    exe_name="mf6",
    version="mf6",
    sim_ws=gwf_ws,
)
tdis = flopy.mf6.ModflowTdis(
    gwf_sim,
    time_units=time_unit,
    nper=nper,
    perioddata=tdis_pd,
)
gwf = flopy.mf6.ModflowGwf(
    gwf_sim,
    modelname=gwf_name,
    save_flows=True,
)
dis = flopy.mf6.ModflowGwfdis(
    gwf,
    length_units=length_unit,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)
ic = flopy.mf6.ModflowGwfic(gwf, strt=top)

#### Create the Node Property Flow (NPF) package

Set the aquifer's hydraulic properties with `flopy.mf6.ModflowGwfnpf()`. Make the top layer convertible (`icelltype=1`, unconfined) and the bottom two confined (`icelltype=0`). Horizontal conductivity (`k`) is larger than vertical (`k33`). Set `save_specific_discharge=True` so the particle-tracking models later have the flow directions they need.

In [ ]:
kh = [50.0, 0.01, 200.0]
kv = [10.0, 0.01, 20.0]
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=[1, 0, 0],
    k=kh,
    k33=kv,
    save_flows=True,
    save_saturation=True,
    save_specific_discharge=True,
)

#### Define the boundary conditions

Add the pumping well with `flopy.mf6.ModflowGwfwel()`. This is the first **list-based** stress package: its data are a list of tuples, one per boundary cell, passed through `stress_period_data` keyed by the zero-based stress-period number. Each cell ID (`cellid`) is a `(layer, row, column)` tuple counting from 0. For the well package each tuple is the cell ID followed by the pumping rate `q` (negative for withdrawal):

```python
# (cellid, q) = ((layer, row, column), rate)
((2, 10, 9), -150000.0)
```

The well sits in the bottom layer and pumps at `wel_q`.

In [ ]:
wel_q = -150000.0
wel_loc = (2, 10, 9)
wel_pd = [(wel_loc, wel_q)]
well = flopy.mf6.modflow.mfgwfwel.ModflowGwfwel(
    gwf, maxbound=1, stress_period_data={0: wel_pd}
)

Add the river (**RIV**) package along the grid's right edge with `flopy.mf6.ModflowGwfriv()`. This is the first place MP7 and PRT differ. By default a boundary condition acts as a distributed source or sink spread over the whole cell. To instead attach the flow to a specific **cell face** — which changes where particles enter or leave — use MODFLOW 6's auxiliary-variable mechanism. MP7 reads a face number from an auxiliary variable named `IFACE`; PRT reads its own `IFLOWFACE`, using a different face-numbering scheme. Here you set both so the same flow model drives either code, routing the river flow through the cells' top face. The list comprehension builds one river tuple per row of column `ncol - 1`; the inline comment shows the tuple field order.

In [ ]:
riv_h = 320.0
riv_z = 317.0
riv_c = 1.0e5
riv_iface = 6
riv_iflowface = -1
riv_pd = []
for i in range(nrow):
    riv_pd.append(
        [
            # (layer, row, column), stage, conductance, bottom, [aux, ...]
            (0, i, ncol - 1),
            riv_h,
            riv_c,
            riv_z,
            riv_iface,
            riv_iflowface,
        ]
    )
riv = flopy.mf6.modflow.mfgwfriv.ModflowGwfriv(
    gwf, auxiliary=["iface", "iflowface"], stress_period_data={0: riv_pd}
)

Add areal recharge (**RCH**) across the whole top of the model with `flopy.mf6.ModflowGwfrcha()`. Use the same `IFACE`/`IFLOWFACE` values as the river so recharge enters through the cells' top face.

In [ ]:
rch = 0.005
rch_iface = 6
rch_iflowface = -1
rcha = flopy.mf6.modflow.mfgwfrcha.ModflowGwfrcha(
    gwf,
    recharge=rch,
    auxiliary=["iface", "iflowface"],
    aux=[rch_iface, rch_iflowface],
)

#### Configure output control and the solver

Set up the output control (**OC**) package with `flopy.mf6.ModflowGwfoc()`. Save head to the `.hds` file and the cell-by-cell flow budget to the `.cbb` file for every time step — the particle-tracking models read both of these files.

In [ ]:
headfile_name = f"{gwf_name}.hds"
budgetfile_name = f"{gwf_name}.cbb"

oc = flopy.mf6.modflow.mfgwfoc.ModflowGwfoc(
    gwf,
    pname="oc",
    head_filerecord=[headfile_name],
    budget_filerecord=[budgetfile_name],
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)

Add the iterative model solution (**IMS**) with `flopy.mf6.ModflowIms()` — the numerical solver that the flow model uses to compute heads. The tight close criteria give an accurate head field for tracking.

In [ ]:
ims = flopy.mf6.ModflowIms(
    gwf_sim,
    pname="ims",
    complexity="SIMPLE",
    outer_dvclose=1e-6,
    inner_dvclose=1e-6,
    rcloserecord=1e-6,
)

#### Write and run the flow model

Write the input files with `sim.write_simulation()`, then run MODFLOW 6 with `sim.run_simulation()`. The `assert` stops the notebook if the model does not converge, so you never track particles through a bad flow field.

In [ ]:
gwf_sim.write_simulation(silent=False)
success, buff = gwf_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

#### Load and plot the flow results

Load the heads with `gwf.output.head()` and call `.get_data()` to get the head array. Then map the bottom layer (`layer=2`) with `flopy.plot.PlotMapView()`, drawing the head array with `.plot_array()` and the well and river cells with `.plot_bc()`.

In [ ]:
hds = gwf.output.head()
head = hds.get_data()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(6, 6))
    ax.set_aspect("equal")
    ax.set_title("Head, layer 3", fontsize=10)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="RIV"),
        ],
        loc="upper left",
    )
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=2)
    mm.plot_grid(alpha=0.25)
    mm.plot_bc("WEL", plotAll=True, color="red")
    mm.plot_bc("RIV", plotAll=True, color="blue")
    pc = mm.plot_array(head, alpha=0.25)
    cb = plt.colorbar(pc, shrink=0.25, pad=0.1)
    cb.ax.set_xlabel(r"Head (ft)")
    fig.tight_layout()
    plt.show()

**What to look for.** Head is lowest (the cool colors) at the red well cell, where pumping draws the water table down, and highest along the blue river cells on the right edge. Water moves down the head gradient — from the river toward the well — which is exactly the path the backward particles will retrace in reverse.

## Phase 2 — Track particles with MODPATH 7

With the flow field in hand, build the first tracking model: **MODPATH 7 (MP7)**, the stand-alone tracking code.

Define where particles start with FloPy's MP7 release-template utilities, then later convert them for PRT. Here you release particles from the four lateral faces of the well cell using MP7 particle input style 2, subdivision style 1 — `FaceDataType` says how finely to subdivide each face, and `LRCParticleData` places that pattern on the well cell (identified by its layer-row-column region).

In [ ]:
mp7_face_data = flopy.modpath.FaceDataType(
    verticaldivisions1=5,
    verticaldivisions2=5,
    verticaldivisions3=5,
    verticaldivisions4=5,
    horizontaldivisions1=5,
    horizontaldivisions2=5,
    horizontaldivisions3=5,
    horizontaldivisions4=5,
    rowdivisions5=0,
    rowdivisions6=0,
    columndivisions5=0,
    columndivisions6=0,
)
mp7_particle_data = flopy.modpath.LRCParticleData(
    subdivisiondata=[mp7_face_data], lrcregions=[[[*wel_loc, *wel_loc]]]
)
mp7_pg1 = flopy.modpath.ParticleGroupLRCTemplate(
    particlegroupname="PG1",
    particledata=mp7_particle_data,
    filename="pg1.sloc",
)

Set up the MP7 model with `flopy.modpath.Modpath7()`, pointing it at the flow model (`gwf`) and the head and budget files you just wrote. Set the porosity in the basic (`Bas`) package, then configure the run with `Modpath7Sim()`. The key settings make this a **backward** pathline simulation: `trackingdirection="backward"` traces particles upstream from the well.

In [ ]:
mp7_name = f"{example_name}-mp7"
mp7_ws = base_ws / "mp7"
mp7_ws.mkdir(exist_ok=True, parents=True)

mp7 = flopy.modpath.Modpath7(
    modelname=mp7_name,
    flowmodel=gwf,
    exe_name="mp7",
    model_ws=mp7_ws,
    budgetfilename=budgetfile_name,
    headfilename=headfile_name,
)

porosity = 0.1
mp7_bas = flopy.modpath.Modpath7Bas(
    mp7,
    porosity=porosity,
    defaultiface={"RCH": 6},  # recharge enters through the top cell face
)

mp7_sim = flopy.modpath.Modpath7Sim(
    mp7,
    simulationtype="pathline",
    trackingdirection="backward",
    budgetoutputoption="summary",
    stoptimeoption="extend",
    particlegroups=[mp7_pg1],
)

Write the MP7 input files with `.write_input()` and run the model with `.run_model()`.

In [ ]:
mp7.write_input()
success, buff = mp7.run_model(silent=False)
assert success, "MODPATH 7 did not terminate normally"

#### Load and plot the MP7 pathlines

Read the pathline output with FloPy's `PathlineFile` utility. Its `.get_alldata()` returns a list of NumPy recarrays, one per particle. Use a list comprehension to turn each recarray into a pandas `DataFrame` and `pd.concat()` to stack them into a single table for easy inspection and plotting.

In [ ]:
import pandas as pd

mp7_pathline_file = flopy.utils.PathlineFile(mp7_ws / f"{mp7_name}.mppth")
mp7_pathlines = pd.concat(
    [pd.DataFrame(pdata) for pdata in mp7_pathline_file.get_alldata()]
)
mp7_pathlines

Plot the MP7 pathlines over the head field with `.plot_pathline()`.

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(10, 5))
    ax.set_aspect("equal")
    ax.set_title("MODPATH 7 pathlines", fontsize=11)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="RIV"),
        ],
        loc="upper left",
    )
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=2)
    mm.plot_grid(alpha=0.2)
    mm.plot_bc("WEL", plotAll=True, color="red")
    mm.plot_bc("RIV", plotAll=True, color="blue")
    pc = mm.plot_array(head, alpha=0.25)
    cb = plt.colorbar(pc, shrink=0.25, pad=0.1)
    mm.plot_pathline(mp7_pathlines, layer="all")
    cb.ax.set_xlabel(r"Head (ft)")
    fig.tight_layout()
    plt.show()

**What to look for.** Each line is one particle's path from the well (red) back out into the aquifer. Because tracking runs backward, the lines fan out from the well toward the areas that supply its water — many reach the river on the right. The envelope of all the lines outlines the well's capture zone; you will quantify that area with the PRT results below.

## Phase 3 — Track the same particles with MODFLOW 6 PRT

Now build an **equivalent PRT model** to reproduce the MP7 result inside MODFLOW 6. A PRT model lives in its own MODFLOW 6 simulation, so start the same way as the flow model — simulation, TDIS, then the PRT model (`flopy.mf6.ModflowPrt()`) with a **DIS** grid that matches the flow model. Add the Model Input Parameters (**MIP**) package with `flopy.mf6.ModflowPrtmip()` to set the porosity, which controls how fast particles move.

In [ ]:
prt_name = f"{example_name}-prt"
prt_ws = base_ws / "prt"
prt_ws.mkdir(exist_ok=True, parents=True)

prt_sim = flopy.mf6.MFSimulation(
    sim_name=prt_name, exe_name="mf6", version="mf6", sim_ws=prt_ws
)

tdis = flopy.mf6.modflow.mftdis.ModflowTdis(
    prt_sim,
    pname="tdis",
    time_units="DAYS",
    nper=nper,
    perioddata=[(perlen, nstp, tsmult)],
)

prt = flopy.mf6.ModflowPrt(
    prt_sim, modelname=prt_name, model_nam_file=f"{prt_name}.name"
)

dis = flopy.mf6.modflow.mfgwfdis.ModflowGwfdis(
    prt,
    pname="dis",
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    length_units="FEET",
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)

mip = flopy.mf6.ModflowPrtmip(prt, pname="mip", porosity=porosity)

Define the release points with the particle release point (**PRP**) package, `flopy.mf6.ModflowPrtprp()`. PRT does not yet support MP7-style release templates, so every release point must be listed explicitly. Reuse the MP7 particle data and convert it with `.to_prp()`, which returns exactly those points. `perioddata={0: ["FIRST"]}` releases all particles at the start of the first stress period.

In [ ]:
release_pts = list(mp7_particle_data.to_prp(gwf.modelgrid))
prp = flopy.mf6.ModflowPrtprp(
    prt,
    nreleasepts=len(release_pts),
    packagedata=release_pts,
    perioddata={0: ["FIRST"]},
    exit_solve_tolerance=1e-5,
    extend_tracking=True,
)

Create the output control (**OC**) package with `flopy.mf6.ModflowPrtoc()`. PRT can write track output as binary or CSV; use CSV here.

Where MP7 has four separate simulation modes (pathline, timeseries, endpoint, combined), PRT has one mode and instead reports **tracking events**, which you enable or disable. Each event carries a **reason code** in the `ireason` column of the CSV:

- 0: a particle was released
- 1: a particle exited a grid feature (e.g. a cell)
- 2: a time step ended
- 3: a particle terminated
- 4: a particle entered a weak sink cell (a cell that removes some but not all inflow)
- 5: the simulation reached a user-specified tracking time
- 6: a particle was dropped to the water table (covered in the next notebook)

By default every event is reported. You will use reason code 3 (termination) below to find where each backward particle ended up. (To mimic an MP7 endpoint run, report only release and termination events; to mimic a timeseries run, report time events and supply tracking times.)

In [ ]:
budgetfile_prt = f"{prt_name}.cbb"
trackcsvfile_prt = f"{prt_name}.trk.csv"

oc = flopy.mf6.ModflowPrtoc(
    prt,
    pname="oc",
    budget_filerecord=[budgetfile_prt],
    trackcsv_filerecord=[trackcsvfile_prt],
    saverecord=[("BUDGET", "ALL")],
)

#### Create the flow model interface and feed it reversed flows

Connect the PRT model to the flow results with the flow model interface (**FMI**) package, `flopy.mf6.ModflowPrtfmi()`. PRT does not yet track backward on its own, so instead you hand it a **time-reversed** copy of the flow output: load the head and budget files and call `.reverse()`, which flips the time order of the records and switches the sign of the flows. Tracking forward through the reversed field is equivalent to tracking backward through the original.

In [ ]:
head_file = flopy.utils.HeadFile(gwf_ws / headfile_name, tdis=gwf_sim.tdis)
budget_file = flopy.utils.CellBudgetFile(
    gwf_ws / budgetfile_name, precision="double", tdis=gwf_sim.tdis
)

headfile_bkwd_name = f"{headfile_name}_bkwd"
budgetfile_bkwd_name = f"{budgetfile_name}_bkwd"

head_file.reverse(prt_ws / headfile_bkwd_name)
budget_file.reverse(prt_ws / budgetfile_bkwd_name)

fmi = flopy.mf6.ModflowPrtfmi(
    prt,
    packagedata=[
        ("GWFHEAD", headfile_bkwd_name),
        ("GWFBUDGET", budgetfile_bkwd_name),
    ],
)

Add the explicit model solution (**EMS**) with `flopy.mf6.ModflowEms()` and register it to the PRT model. Unlike flow and transport, PRT does not solve a matrix — it moves particles cell by cell — so it uses this lightweight explicit solution rather than an IMS.

In [ ]:
ems = flopy.mf6.ModflowEms(
    prt_sim,
    pname="ems",
    filename=f"{prt_name}.ems",
)
prt_sim.register_solution_package(ems, [prt.name])

Write and run the PRT simulation.

In [ ]:
prt_sim.write_simulation()
success, buff = prt_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

Load the PRT pathlines. Because you chose CSV output, read them straight into a DataFrame with `pd.read_csv()`.

In [ ]:
prt_pathlines = pd.read_csv(prt_ws / trackcsvfile_prt)
prt_pathlines

#### Compare MP7 and PRT in 3D

Overlay the two sets of pathlines in three dimensions with **PyVista** to confirm they agree. Build a VTK mesh of the grid and the tracks with FloPy's `Vtk` exporter, add boxes marking the river, well, and confining bed, and render everything together. The MP7 tracks are drawn in purple, the PRT tracks in pink.

In [ ]:
import pyvista as pv
from flopy.export.vtk import Vtk

vert_exag = 10

vtk = Vtk(model=gwf, binary=False, vertical_exageration=vert_exag, smooth=False)
vtk.add_model(gwf)
vtk.add_pathline_points(mp7_pathlines.to_records(index=False))
gwf_mesh, mp7_mesh = vtk.to_pyvista()

# FloPy's Vtk class only supports one pathline set at a time, so rebuild for PRT
vtk = Vtk(model=gwf, binary=False, vertical_exageration=vert_exag, smooth=False)
vtk.add_model(gwf)
vtk.add_pathline_points(prt_pathlines.to_records(index=False))
_, prt_mesh = vtk.to_pyvista()


def get_nn(k, i, j):
    return k * nrow * ncol + i * ncol + j


riv_mesh = pv.Box(
    bounds=[
        gwf.modelgrid.extent[1] - delc,
        gwf.modelgrid.extent[1],
        gwf.modelgrid.extent[2],
        gwf.modelgrid.extent[3],
        bot[0] * vert_exag,
        head[(0, 0, ncol - 1)] * vert_exag,
    ]
)
well_cellid = get_nn(0, *wel_loc[1:])
well_points = gwf.modelgrid.verts[gwf.modelgrid.iverts[well_cellid]]
well_xs, well_ys = list(zip(*well_points, strict=False))
wel_mesh = pv.Box(
    bounds=[
        min(well_xs),
        max(well_xs),
        min(well_ys),
        max(well_ys),
        bot[-1] * vert_exag,
        bot[-2] * vert_exag,
    ]
)
bed_mesh = pv.Box(
    bounds=[
        gwf.modelgrid.extent[0],
        gwf.modelgrid.extent[1],
        gwf.modelgrid.extent[2],
        gwf.modelgrid.extent[3],
        bot[1] * vert_exag,
        bot[0] * vert_exag,
    ]
)

pv.set_jupyter_backend("static")
p = pv.Plotter(window_size=[500, 500], notebook=True)
p.enable_anti_aliasing()
p.add_mesh(gwf_mesh, opacity=0.025, style="wireframe")
p.add_mesh(mp7_mesh, point_size=5, line_width=2.5, smooth_shading=True, color="purple")
p.add_mesh(prt_mesh, point_size=5, line_width=2.5, smooth_shading=True, color="pink")
p.add_mesh(riv_mesh, color="teal", opacity=0.2)
p.add_mesh(wel_mesh, color="red", opacity=0.3)
p.add_mesh(bed_mesh, color="tan", opacity=0.1)
p.camera.elevation -= 15
p.show()

**What to look for.** The purple (MP7) and pink (PRT) tracks lie essentially on top of one another — the two codes produce the same paths from the same flow field, which is the point of the migration. You can also see the vertical structure the map view hid: particles rise from the deep well (red box), cross the tan confining bed, and travel through the upper aquifer toward the river (teal box).

#### Estimate the capture zone

Estimate the well's capture area from the particle endpoints. First filter the pathlines to the **termination** events (`ireason == 3`) — one per particle, giving where each backward track ended. The `assert` checks you have one terminal point per release point.

In [ ]:
terminals = prt_pathlines[prt_pathlines.ireason == 3]
assert len(terminals) == len(release_pts)

Wrap the endpoints in a Shapely `MultiPoint` and take their **convex hull** — the smallest polygon enclosing them all — as an estimate of the capture zone. Convert its area from square feet to square miles.

In [ ]:
from shapely import MultiPoint, convex_hull

term_pts = MultiPoint(terminals[["x", "y", "z"]].to_numpy())
chull = convex_hull(term_pts)
chull_area_mi = chull.area / (5280 * 5280)
print(f"Capture area: {chull.area:9.3f} sq {length_unit}, {chull_area_mi:9.3f} sq mi")
chull

Plot the PRT pathlines with the capture-zone polygon shaded on top.

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(6, 6))
    ax.set_aspect("equal")
    fig.tight_layout()
    ax.set_title("PRT pathlines with capture zone", fontsize=11)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="RIV"),
            Patch(color="teal", label="capture zone"),
        ],
        loc="upper left",
    )
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=2)
    mm.plot_grid(alpha=0.2)
    mm.plot_bc("WEL", plotAll=True, color="red")
    mm.plot_bc("RIV", plotAll=True, color="blue")
    pc = mm.plot_array(head, alpha=0.25)
    mm.plot_pathline(prt_pathlines, layer="all")
    chull_xs, chull_ys, _ = map(list, zip(*list(chull.exterior.coords), strict=False))
    ax.fill(chull_xs, chull_ys, "teal", alpha=0.5)
    ax.annotate(f"{chull_area_mi:.3f} sq mi", (1000, 3000), color="red")
    plt.show()

**What to look for.** The shaded teal polygon is the estimated capture zone — every particle path stays inside it, and the annotated area (in square miles) is your capture-area estimate. Water falling as recharge inside this footprint eventually reaches the well; water outside it does not. Because the estimate is a convex hull of the endpoints, it slightly over-covers any concave notches in the true capture area.

#### Examine travel times

The `t` column of each terminal record is the particle's total travel time — how long water takes to reach the well. Summarize the distribution with `.describe()`, then plot it as a histogram.

In [ ]:
terminals.t.describe()

In [ ]:
terminals.t.plot(kind="hist")
plt.show()

**What to look for.** The histogram shows the spread of travel times to the well. Most particles reach it in under roughly 30 years (about 11,000 days), with a tail of longer paths — the water arriving at the well is a mixture of many ages, not a single travel time.

#### Map travel time back to location

Finally, put travel time back in space. Redraw the endpoints on the map, coloring each by its travel time `t` with `ax.scatter()`.

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(6, 6))
    ax.set_aspect("equal")
    fig.tight_layout()
    ax.set_title("PRT endpoints colored by travel time", fontsize=11)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="RIV"),
        ],
        loc="upper left",
    )
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=2)
    mm.plot_grid(alpha=0.2)
    mm.plot_bc("WEL", plotAll=True, color="red")
    mm.plot_bc("RIV", plotAll=True, color="blue")
    pc = mm.plot_array(head, alpha=0.25)
    pc = ax.scatter(terminals.x, terminals.y, c=terminals.t)
    cb = plt.colorbar(pc, shrink=0.25, pad=0.1)
    cb.ax.set_xlabel(r"Travel time ($d$)")
    plt.show()

**What to look for.** Each dot is where a backward particle ended, colored by how long its water takes to reach the well. Endpoints near the well are the coolest (shortest travel times); those farther out — toward the river and the up-gradient recharge area — are warmer, because that water travels farther and arrives later. Together the color pattern maps not just *where* the well's water comes from but *how old* it is.

## Recap

In this notebook you:

- built and ran a three-layer steady-state **groundwater-flow (GWF)** model, using cell-face auxiliary variables (`IFACE`/`IFLOWFACE`) so the river and recharge flows enter through the correct face;
- ran **backward particle tracking** two ways from that one flow field — first with stand-alone **MODPATH 7 (MP7)**, then with MODFLOW 6's built-in **particle-tracking (PRT)** model — feeding PRT a time-reversed copy of the flow output through the flow model interface (**FMI**) package;
- confirmed the two codes agree by overlaying their pathlines in 3D; and
- post-processed the PRT endpoints into a **capture-zone** area (via a convex hull) and a **travel-time** distribution, then mapped travel time back to location.

The next notebook builds on the PRT tracking-event and water-table ideas introduced here.